In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision  # has different datasets and pretrained CNNs and utilities for image transformation
from torchvision.datasets import CIFAR10

In [7]:
### Datasets & Dataloaders

from torch.utils.data import DataLoader
import torchvision.transforms as transforms  # used to perform transformations

# Scaling (0,1) & Normalization (-1,1)

transform = transforms.Compose(  # combines multiple tranformation and apply one after other
    [
        transforms.ToTensor(), # convert images into tensors and apply scaling 
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # Normalization is done because , Neural networks train faster when  inputs as centered around Zero
    ]
)

trainset = CIFAR10(root="./data" , train=True , download=True ,transform=transform)
testset=CIFAR10(root="./data" , train=False , download=True ,transform=transform)

In [8]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

In [12]:
image, label = trainset[0]
print(image.size())

torch.Size([3, 32, 32])


# CNN

In [16]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            ##Layer 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # kernel size , stride
            ##Layer 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            ##Layer 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256), nn.ReLU(), nn.Linear(256,10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # To flatten
        x = self.fc_layers(x)

        return x

In [17]:
model = CNN()

In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the CNN

In [20]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:

        optimizer.zero_grad()

        output = model.forward(images)  # FP
        loss = criterion(output, labels)  # loss func
        loss.backward()  # BP
        optimizer.step()  # update parameters

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")

epoch = 1/10 & loss = 0.8259821775776651
epoch = 2/10 & loss = 0.6750210231679785
epoch = 3/10 & loss = 0.5652105268996085
epoch = 4/10 & loss = 0.4630621757615558
epoch = 5/10 & loss = 0.3713663705550801
epoch = 6/10 & loss = 0.29339962617477494
epoch = 7/10 & loss = 0.22258857765313608
epoch = 8/10 & loss = 0.16860271556793577
epoch = 9/10 & loss = 0.13032634973125842
epoch = 10/10 & loss = 0.11732121814837884


In [22]:
# Evaluation
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images,labels in testloader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels +=labels.size(0)

print(f"Accuracy : {correct_labels/total_labels*100} %")

Accuracy : 75.28 %
